# LLM + Knowledge Graphs

## LLM + Knowledge Graphs

LLMs generate fluent text but may hallucinate facts. Grounding LLM outputs in a knowledge graph provides factual verification.

**Approach**: After LLM generates a claim, extract entities and relations, query a KG for verification, flag unverified or contradicted claims.

**Entity linking**: Map mentions in text to canonical KG entities (e.g., 'the Eiffel Tower' -> Q243 in Wikidata).
**Relation verification**: Given (subject, relation, object), check if this triple exists in the KG or can be inferred.
**SPARQL**: The standard query language for RDF knowledge graphs. LLMs can generate SPARQL queries from natural language questions.

In [ ]:
from typing import Dict, List, Tuple
import re

# Simulate a small knowledge graph for factual grounding
class SimpleKnowledgeGraph:
    def __init__(self):
        self.triples = set()
        self.aliases = {}  # entity aliases
    
    def add(self, s, p, o): self.triples.add((s.lower(), p.lower(), o.lower()))
    def add_alias(self, alias, canonical): self.aliases[alias.lower()] = canonical.lower()
    def resolve(self, entity): return self.aliases.get(entity.lower(), entity.lower())
    
    def verify(self, s, p, o):
        rs, rp, ro = self.resolve(s), p.lower(), self.resolve(o)
        return (rs, rp, ro) in self.triples
    
    def query_object(self, s, p):
        rs, rp = self.resolve(s), p.lower()
        return [o for ss, pp, o in self.triples if ss == rs and pp == rp]

# Build a world knowledge graph
kg = SimpleKnowledgeGraph()
kg.add('paris', 'capital_of', 'france')
kg.add('france', 'located_in', 'europe')
kg.add('eiffel_tower', 'located_in', 'paris')
kg.add('eiffel_tower', 'built_by', 'gustave_eiffel')
kg.add('einstein', 'born_in', 'ulm')
kg.add('einstein', 'nationality', 'german')
kg.add('python', 'created_by', 'guido_van_rossum')
kg.add_alias('the eiffel tower', 'eiffel_tower')
kg.add_alias('albert einstein', 'einstein')

# LLM-style claim extraction and verification
claims = [
    ('paris', 'capital_of', 'france'),
    ('einstein', 'born_in', 'berlin'),  # wrong
    ('eiffel_tower', 'located_in', 'paris'),
    ('python', 'created_by', 'guido_van_rossum'),
    ('france', 'located_in', 'asia'),  # wrong
]

print('LLM Claim Verification against Knowledge Graph:')
for s, p, o in claims:
    verified = kg.verify(s, p, o)
    status = 'VERIFIED' if verified else 'UNVERIFIED/FALSE'
    print(f'  [{status}] ({s}, {p}, {o})')

print('\nKG-grounded answers:')
for q in [('paris', 'capital_of'), ('einstein', 'born_in')]:
    answers = kg.query_object(*q)
    print(f'  {q[0]} {q[1]}: {answers}')
